<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/Evolutionary_Algorithm_Sphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math

def fitness_sphere(sol):
    return sum(x**2 for x in sol)

def generate_solution_sphere(n, min_val=-5.12, max_val=5.12):
    return [random.uniform(min_val, max_val) for _ in range(n)]

def generate_rnd_population_sphere(population_size, problem_size):
    return [generate_solution_sphere(problem_size) for _ in range(population_size)]

def selection_sphere(population,fitness):
    indices = random.sample(range(len(population)), 10)

    pool1 = [population[i] for i in indices[:5]]
    pool1.sort(key=fitness)
    parent1 = pool1[0]
    pool2 = [population[i] for i in indices[5:]]
    pool2.sort(key=fitness)
    parent2 = pool2[0]

    return parent1, parent2

def arithmetic_crossover(population_size, population, crossover_count):
    children = []
    alpha = 0.5
    for _ in range(crossover_count):
        p1, p2 = selection_sphere(population,fitness_sphere)
        child1 = [alpha * x1 + (1 - alpha) * x2 for x1, x2 in zip(p1, p2)]
        child2 = [(1 - alpha) * x1 + alpha * x2 for x1, x2 in zip(p1, p2)]
        children.extend([child1, child2])
    return children

def gaussian_mutation(children, problem_size, mutation_rate, sigma=0.1):
    mutated_children = []
    for child in children:
        new_child = list(child)
        if random.random() < mutation_rate:
            idx = random.randint(0, problem_size - 1)
            new_child[idx] += random.gauss(0, sigma)
            new_child[idx] = max(min(new_child[idx], 5.12), -5.12)
        mutated_children.append(new_child)
    return mutated_children

def new_population_sphere(old_population, mutated_children, population_size):
    combined = old_population + mutated_children
    combined.sort(key=fitness_sphere)
    return combined[:population_size]

def evaluate_population_sphere(population):
    fits = [fitness_sphere(ind) for ind in population]
    fits.sort()
    return fits[0], sum(fits)/len(fits), fits[-1]

In [ ]:
def evolutionary_algorithm_sphere(population_size: int, mutation_rate: float,
                                 crossover_rate: float, generations: int,
                                 problem_size: int):

    population = generate_rnd_population_sphere(population_size, problem_size)


    b, a, w = evaluate_population_sphere(population)
    best_history = [b]
    avg_history = [a]
    worst_history = [w]

    crossover_count = int((population_size * crossover_rate) // 2)


    for i in range(generations):
        children = arithmetic_crossover(population_size, population, crossover_count)
        mutants = gaussian_mutation(children, problem_size, mutation_rate)
        population = new_population_sphere(population, mutants, population_size)
        b, a, w = evaluate_population_sphere(population)
        best_history.append(b)
        avg_history.append(a)
        worst_history.append(w)

    return best_history, avg_history, worst_history

In [ ]:
n_dimensions = 5
best_h, avg_h, worst_h = evolutionary_algorithm_sphere(
    population_size=200,
    mutation_rate=0.1,
    crossover_rate=0.8,
    generations=500,
    problem_size=n_dimensions
)

print(f"Final Best Fitness: {best_h[-1]:.10f}")


Final Best Fitness: 0.0000000487


We will extend the Evolutionary Algorithm to optimize both the **Sphere Function** and the **Schwefel Function**. To satisfy the requirements, we will implement and compare two selection strategies, two crossover operators, and two mutation operators.


**1. Problem Definitions**

**Sphere Function** ($f_1$): A continuous, convex, unimodal function where $f(x) = \sum x_i^2$. The goal is to reach $f(x)=0$ at $x_i=0$.

**Schwefel Function** ($f_7$): A deceptive, multimodal function where $f(x) = \sum -x_i \cdot \sin(\sqrt{|x_i|})$. It is difficult because the global minimum is geometrically distant from the next best local minima.

Search Space: $-500 \le x_i \le 500$.

Global Optimum: $f(x) = -n \cdot 418.9829$ at $x_i = 420.9687$.

In [ ]:
def fitness_schwefel(sol):
  return sum(-x * math.sin(math.sqrt(abs(x))) for x in sol)

**2. Implementing the Comparative Operators**

**A. Selection Strategies**

**1. Tournament Selection**: Picks the best from a random subset.

**2. Roulette Wheel Selection**: Picks parents with a probability proportional to their fitness.

In [ ]:
def roulette_wheel_selection(population, fitnesses):
  max_fit = max(fitnesses)
  probs = [(max_fit - f) + 1e-6 for f in fitnesses]
  total_prob = sum(probs)
  pick = random.uniform(0,total_prob)
  current = 0
  for i, p in enumerate(probs):
    current += p
    if current > pick:
      return population[i]
  return population[-1]

**B. Crossover Operators**

**Arithmetic Crossover**: $Child = \alpha P_1 + (1-\alpha)P_2$.

**Blend Crossover** (BLX-$\alpha$): Picks a random value in the range $[min - \alpha \cdot diff, max + \alpha \cdot diff]$.

In [ ]:
def blx_alpha_crossover(p1, p2,low_bound,high_bound, alpha=0.5):
  child1, child2 = [], []
  for x1,x2 in zip(p1,p2):
    diff = abs(x1-x2)
    low = min(x1, x2) - alpha * diff
    high = max(x1,x2) + alpha * diff
    v1 = max(low_bound, min(random.uniform(low, high), high_bound))
    v2 = max(low_bound, min(random.uniform(low, high), high_bound))
    child1.append(v1)
    child2.append(v2)
  return child1, child2

**C. Mutation Operators**

**Gaussian Mutation**: Adds noise from a Normal distribution.

**Uniform Mutation**: Replaces a gene with a random value from the valid range.

In [ ]:
def uniform_mutation(individual, prob, low, high):
    for i in range(len(individual)):
        if random.random() < prob:
            individual[i] = random.uniform(low, high)
    return individual

**3. Comparative Experiment Script**

In [ ]:
import csv

def evolutionary_algorithm_A6(population_size, mutation_rate, crossover_rate,
                             generations, problem_size, fitness_func,
                             selection_type="Tournament", crossover_type="Arithmetic",
                             mutation_type="Gaussian", bounds=(-5.12, 5.12)):

    low, high = bounds
    population = [[random.uniform(low, high) for _ in range(problem_size)] for _ in range(population_size)]

    fits = [fitness_func(ind) for ind in population]
    best_history = [min(fits)]
    avg_history = [sum(fits)/len(fits)]
    worst_history = [max(fits)]

    crossover_count = int((population_size * crossover_rate) // 2)

    for gen in range(generations):
        children = []

        for _ in range(crossover_count):
            if selection_type == "Tournament":
                p1, p2 = selection_sphere(population,fitness_func)
            else:
                current_fits = [fitness_func(ind) for ind in population]
                p1 = roulette_wheel_selection(population, current_fits)
                p2 = roulette_wheel_selection(population, current_fits)

            if crossover_type == "Arithmetic":
                alpha = 0.5
                c1 = [alpha * x1 + (1 - alpha) * x2 for x1, x2 in zip(p1, p2)]
                c2 = [(1 - alpha) * x1 + alpha * x2 for x1, x2 in zip(p1, p2)]
            else:
                c1, c2 = blx_alpha_crossover(p1, p2,low,high)
            children.extend([c1, c2])

        mutants = []
        for child in children:
            if mutation_type == "Gaussian":
                m_child = list(child)
                if random.random() < mutation_rate:
                    idx = random.randint(0, problem_size - 1)
                    if fitness_func == fitness_sphere:
                      m_child[idx] += random.gauss(0, 0.1)
                    else:
                      m_child[idx] += random.gauss(0, 25)
                    m_child[idx] = max(min(m_child[idx], high), low)
                mutants.append(m_child)
            else:
                mutants.append(uniform_mutation(list(child), mutation_rate, low, high))

        combined = population + mutants
        combined.sort(key=fitness_func)
        population = combined[:population_size]

        current_fits = [fitness_func(ind) for ind in population]
        best_history.append(min(current_fits))
        avg_history.append(sum(current_fits)/len(current_fits))
        worst_history.append(max(current_fits))

    return best_history, avg_history, worst_history



In [ ]:
pop = 50
gens = 100
dim = 5
mut_p = 0.1
cross_p = 0.8
num_runs = 8

problems = [
    {"name": "Sphere", "func": fitness_sphere, "bounds": (-5.12, 5.12)},
    {"name": "Schwefel", "func": fitness_schwefel, "bounds": (-500, 500)} #
]

results = []

for prob in problems:
    run_results = []
    for _ in range(num_runs):
        bh, _, _ = evolutionary_algorithm_A6(pop, mut_p, cross_p, gens, dim,
                                             prob["func"], "Tournament", "Arithmetic", "Gaussian", prob["bounds"])
        run_results.append(bh[-1])
    results.append([prob["name"], "Tournament", "Arithmetic", "Gaussian", min(run_results), sum(run_results)/num_runs])

    run_results = []
    for _ in range(num_runs):
        bh, _, _ = evolutionary_algorithm_A6(pop, mut_p, cross_p, gens, dim,
                                             prob["func"], "Roulette", "BLX", "Uniform", prob["bounds"])
        run_results.append(bh[-1])
    results.append([prob["name"], "Roulette", "BLX", "Uniform", min(run_results), sum(run_results)/num_runs])

with open('A6_Experiments.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Problem", "Selection", "Crossover", "Mutation", "Best_Fitness", "Avg_Fitness"])
    writer.writerows(results)

print("Experiments complete. Data saved to A6_Experiments.csv")

Experiments complete. Data saved to A6_Experiments.csv


|Problem|Selection|Crossover|Mutation|Best\_Fitness|Avg\_Fitness|
|---|---|---|---|---|---|
|Sphere|Tournament|Arithmetic|Gaussian|2\.1361297889327707e-06|1\.9628288439303667e-05|
|Sphere|Roulette|BLX|Uniform|2\.4060951973000817e-12|7\.207069497558601e-08|
|Schwefel|Tournament|Arithmetic|Gaussian|-1640\.8892300934858|-1165\.5296694325302|
|Schwefel|Roulette|BLX|Uniform|-2094\.9144363413766|-2093\.9454312120847|


**Results on Sphere Function**


Both configurations achieved values very close to the global optimum.


However, Roulette + BLX + Uniform significantly outperformed Tournament + Arithmetic + Gaussian:

Best fitness improved from ~10^−6
 to ~10^−12

Average fitness also improved by several orders of magnitude.

**Results on Schwefel Function**

f(x)=−n⋅418.9829

For n=5, the optimum is approximately **-2094.9145**.

Roulette + BLX + Uniform reached almost the exact global optimum:

**Best ≈ -2094.91**

**Average ≈ -2093.95** (very close → stable convergence)

Tournament + Arithmetic + Gaussian performed significantly worse:

**Best ≈ -1640.89**

**Average ≈ -1165.53** (large gap → unstable and premature convergence)

Schwefel is highly multimodal and deceptive, with many local minima.

The combination Roulette + BLX + Uniform:

Maintains diversity (roulette selection)

Explores larger regions (BLX crossover)

Introduces strong randomness (uniform mutation)
→ This makes it well-suited for escaping local minima and locating the global optimum.

In contrast, Tournament + Arithmetic + Gaussian:

Encourages exploitation and fast convergence

But tends to get trapped in local minima due to insufficient exploration

**Conclusion**

The experimental results demonstrate that:

**Roulette selection + BLX crossover + Uniform mutation** provides superior performance, especially for difficult multimodal problems like Schwefel.


**Tournament selection + Arithmetic crossover + Gaussian mutation** is more suitable for simpler problems but may fail on complex landscapes due to premature convergence.
